In [ ]:

import base64, json, os, pathlib, signal, subprocess, sys, time, torch
T0 = time.monotonic(); CAP = 2700.0; CUT = 900.0
w = pathlib.Path("/kaggle/working"); p = w / "payload"; p.mkdir(exist_ok=True)
BLOB = {'piezo_pinn.py': 'IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMw0KIiIiRFJBRlRfVU5WRVJJRklFRDogaW5kaXJlY3QgKHZvbHRhZ2UtZHJpdmVuKSBQVkRGIGJpbW9ycGgsIDJEIHBsYW5lIHN0cmVzcywgbWl4ZWQgOC1vdXRwdXQgUElOTiAodSx2LHBoaSxzeCxzeix0eHosRHgsRHkpLg0KDQpPd24gaW1wbGVtZW50YXRpb24gb2YgdGhlIHBoeXNpY3MgKG5vdCBEYW5pZWwgR29uemFsZXoncyBjb2RlOiBsaWNlbmNlIHVuY29uZmlybWVkKS4gUHJvYmxlbSBhcyBpbiB0aGUgdGhlc2lzIFA1IC8gRGFuaWVsJ3MgaW5kaXJlY3QgZm9ybXVsYXRpb246DQpiZWFtIEw9MC4xIG0sIEg9MSBtbSwgY2xhbXBlZCBhdCB4PTAgKHU9dj0wKSwgMTAwIFYgb24gdG9wICh5PUgpLCAwIFYgb24gYm90dG9tICh5PTApLCBvcHBvc2l0ZSBwb2xpbmcgaW4gdGhlIHR3byBsYXllcnMgKGludGVyZmFjZSB5PUgvMikuDQpDb2VmZmljaWVudCBjb252ZW50aW9uIGZpeGVkIGJ5IGBzaWduX2NoZWNrLnB5YCAodGhlIEZFTSBmaWVsZCBzYXRpc2ZpZXMgZGl2KHNpZ21hKT0wLCBkaXYoRCk9MCBvbmx5IHdpdGggb3Bwb3NpdGUtc2lnbiBlMzEvZTMzIGFuZCB0aGUNCmNvbnN0YW50LXN0cmFpbiBwZXJtaXR0aXZpdHkpOiBib3R0b20gbGF5ZXIgZTMxPSswLjAyOTA0LCBlMzM9LTAuMDUxNTg7IHRvcCBsYXllciBmbGlwcGVkLg0KUmVmZXJlbmNlIGZpZWxkOiBEYW5pZWwncyBGRU0gd2l0aCBwb2xpbmdfc2lnbj0tMSAoc2FtZSBjb252ZW50aW9uKSwgMjAxeDIxIGdyaWQuDQpOb25kaW1lbnNpb25hbCBpbnB1dHMgeGk9eC9MLCBldGE9eS9IOyBvdXRwdXRzIHNjYWxlZCBieSBmaXhlZCBjaGFyYWN0ZXJpc3RpYyBzY2FsZXM7IGhhcmQgQkNzIG9uIHUsdixwaGkuDQpBcm1zOiBtbHAgKDItMTAwLTI1MC04IHB5cmFtaWQsIHRhbmgsIEdsb3JvdCkgdnMgcmJmIChSQkYtUElLQU4gd2lkdGggVywgMTEgYmFzZXMvZWRnZSkuIFN0YWdlczogQWRhbSB0aGVuIHN0cm9uZy1Xb2xmZSBMLUJGR1M7IGJvdGggc2NvcmVkLg0KTW9kZXM6IHNtb2tlIHwgcnVuLg0KIiIiDQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zDQoNCmltcG9ydCBhcmdwYXJzZQ0KaW1wb3J0IGpzb24NCmltcG9ydCBtYXRoDQppbXBvcnQgdGltZQ0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQoNCmltcG9ydCBudW1weSBhcyBucA0KaW1wb3J0IHRvcmNoDQpmcm9tIHNjaXB5LnN0YXRzIGltcG9ydCBxbWMNCmZyb20gdG9yY2ggaW1wb3J0IG5uDQoNClNUQVRVUyA9ICJEUkFGVF9VTlZFUklGSUVEIg0KRFQgPSB0b3JjaC5mbG9hdDY0ICAjIHNldCBmcm9tIC0tZHR5cGUNCkRFViA9IHRvcmNoLmRldmljZSgiY3B1IikgICMgc2V0IGZyb20gLS1kZXZpY2UNClZTUExJVCA9IDAuMCAgIyA+MDogdi1oYXQgPSB4aSpbTjEoeGksLjUpICsgVlNQTElUKihOMSh4aSxldGEpLU4xKHhpLC41KSldIChzZXBhcmF0ZXMgdGhlIE8oMSkgYmVuZGluZyBkZWZsZWN0aW9uIGZyb20gdGhlIH4xZS01IHRoaWNrbmVzcyBzdHJhaW4pDQpMLCBILCBWID0gMC4xLCAxZS0zLCAxMDAuMA0KQzExID0gMi4wZTkgLyAoMSAtIDAuMjkgKiogMikNCkMxMiA9IDAuMjkgKiBDMTENCkMyMiA9IEMxMQ0KRyA9IDAuNzc1ZTkNCkQzMSwgRDMzID0gMi4yZS0xMSwgLTMuMGUtMTENCkUzMSA9IEMxMSAqIEQzMSArIEMxMiAqIEQzMw0KRTMzID0gQzEyICogRDMxICsgQzExICogRDMzDQpFUFMgPSAxMiAqIDguODU0MTg3ODE3NjIwZS0xMiAtIChEMzEgKiAoQzExICogRDMxICsgQzEyICogRDMzKSArIEQzMyAqIChDMTIgKiBEMzEgKyBDMTEgKiBEMzMpKSAgIyBjb25zdGFudC1zdHJhaW4gcGVybWl0dGl2aXR5DQpTQ19VLCBTQ19WLCBTQ19TLCBTQ19EID0gM2UtNywgMy4zZS01LCA2ZTMsIDFlLTUgICMgY2hhcmFjdGVyaXN0aWMgc2NhbGVzIGZvciB1LCB2LCBzdHJlc3MsIEQNCg0KDQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIG1vZGVscw0KY2xhc3MgUkJGRWRnZShubi5Nb2R1bGUpOg0KICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpbnB1dHMsIG91dHB1dHMsIGJhc2VzKToNCiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpDQogICAgICAgIHNwYWNpbmcgPSAyLjAgLyAoYmFzZXMgLSAxKQ0KICAgICAgICBzZWxmLnJlZ2lzdGVyX2J1ZmZlcigiY2VudGVycyIsIHRvcmNoLmxpbnNwYWNlKC0xLjAsIDEuMCwgYmFzZXMpKQ0KICAgICAgICBzZWxmLnJlZ2lzdGVyX2J1ZmZlcigid2lkdGgiLCB0b3JjaC50ZW5zb3IoMS43NSAqIHNwYWNpbmcpKQ0KICAgICAgICBzZWxmLmNvZWZmaWNpZW50cyA9IG5uLlBhcmFtZXRlcih0b3JjaC5yYW5kbihvdXRwdXRzLCBpbnB1dHMsIGJhc2VzKSAqICgwLjEgLyBtYXRoLnNxcnQoaW5wdXRzKSkpDQogICAgICAgIHNlbGYuYmFzZSA9IG5uLlBhcmFtZXRlcih0b3JjaC5yYW5kbihvdXRwdXRzLCBpbnB1dHMpICogKDAuMSAvIG1hdGguc3FydChpbnB1dHMpKSkNCg0KICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOg0KICAgICAgICBiYXNpcyA9IHRvcmNoLmV4cCgtKCh4LnVuc3F1ZWV6ZSgtMSkgLSBzZWxmLmNlbnRlcnMpIC8gc2VsZi53aWR0aCkuc3F1YXJlKCkpDQogICAgICAgIHJldHVybiB0b3JjaC5laW5zdW0oImJpZyxvaWctPmJvIiwgYmFzaXMsIHNlbGYuY29lZmZpY2llbnRzKSArIHRvcmNoLm5uLmZ1bmN0aW9uYWwubGluZWFyKHRvcmNoLm5uLmZ1bmN0aW9uYWwuc2lsdSh4KSwgc2VsZi5iYXNlKQ0KDQoNCmNsYXNzIFJCRktBTihubi5Nb2R1bGUpOg0KICAgIGRlZiBfX2luaXRfXyhzZWxmLCBuX2luLCB3aWR0aCwgbl9vdXQsIGJhc2VzPTExKToNCiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpDQogICAgICAgIHNlbGYubGF5ZXJzID0gbm4uTW9kdWxlTGlzdChbUkJGRWRnZShuX2luLCB3aWR0aCwgYmFzZXMpLCBSQkZFZGdlKHdpZHRoLCB3aWR0aCwgYmFzZXMpLCBSQkZFZGdlKHdpZHRoLCBuX291dCwgYmFzZXMpXSkNCg0KICAgIGRlZiBmb3J3YXJkKHNlbGYsIHhpKToNCiAgICAgICAgaCA9IDIuMCAqIHhpIC0gMS4wICAjIGlucHV0cyBhbHJlYWR5IGluIFswLDFdDQogICAgICAgIGZvciBsYXllciBpbiBzZWxmLmxheWVyc1s6LTFdOg0KICAgICAgICAgICAgaCA9IHRvcmNoLnRhbmgobGF5ZXIoaCkpDQogICAgICAgIHJldHVybiBzZWxmLmxheWVyc1stMV0oaCkNCg0KDQpjbGFzcyBNTFAobm4uTW9kdWxlKToNCiAgICBkZWYgX19pbml0X18oc2VsZiwgd2lkdGhzKToNCiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpDQogICAgICAgIHNlbGYubGF5ZXJzID0gbm4uTW9kdWxlTGlzdChubi5MaW5lYXIoYSwgYikgZm9yIGEsIGIgaW4gemlwKHdpZHRoc1s6LTFdLCB3aWR0aHNbMTpdKSkNCiAgICAgICAgZm9yIGxheWVyIGluIHNlbGYubGF5ZXJzOg0KICAgICAgICAgICAgbm4uaW5pdC54YXZpZXJfbm9ybWFsXyhsYXllci53ZWlnaHQpDQogICAgICAgICAgICBubi5pbml0Lnplcm9zXyhsYXllci5iaWFzKQ0KDQogICAgZGVmIGZvcndhcmQoc2VsZiwgeGkpOg0KICAgICAgICB4ID0gMi4wICogeGkgLSAxLjANCiAgICAgICAgZm9yIGxheWVyIGluIHNlbGYubGF5ZXJzWzotMV06DQogICAgICAgICAgICB4ID0gdG9yY2gudGFuaChsYXllcih4KSkNCiAgICAgICAgcmV0dXJuIHNlbGYubGF5ZXJzWy0xXSh4KQ0KDQoNCmNsYXNzIFBpZXpvKG5uLk1vZHVsZSk6DQogICAgIiIiV3JhcHMgYSByYXcgOC1vdXRwdXQgbmV0IHdpdGggaGFyZCBCQ3M7IHJldHVybnMgc2NhbGVkIG91dHB1dHMgKHVoLCB2aCwgcGgsIHN4LCBzeiwgdGF1LCBEeCwgRHkpLiIiIg0KDQogICAgZGVmIF9faW5pdF9fKHNlbGYsIG5ldCk6DQogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQ0KICAgICAgICBzZWxmLm5ldCA9IG5ldA0KDQogICAgZGVmIGZvcndhcmQoc2VsZiwgeGkpOg0KICAgICAgICBvID0gc2VsZi5uZXQoeGkpDQogICAgICAgIHgsIHkgPSB4aVs6LCAwOjFdLCB4aVs6LCAxOjJdDQogICAgICAgIHYgPSBvWzosIDE6Ml0NCiAgICAgICAgaWYgVlNQTElUID4gMDoNCiAgICAgICAgICAgIG1pZCA9IHNlbGYubmV0KHRvcmNoLmNhdChbeCwgdG9yY2guZnVsbF9saWtlKHksIDAuNSldLCBkaW09MSkpWzosIDE6Ml0NCiAgICAgICAgICAgIHYgPSBtaWQgKyBWU1BMSVQgKiAodiAtIG1pZCkNCiAgICAgICAgcmV0dXJuIHRvcmNoLmNhdChbeCAqIG9bOiwgMDoxXSwgeCAqIHYsIHkgKyB5ICogKDEgLSB5KSAqIG9bOiwgMjozXSwgb1s6LCAzOl1dLCBkaW09MSkNCg0KDQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIHByb2JsZW0NCmRlZiBkKGYsIHgpOg0KICAgIHJldHVybiB0b3JjaC5hdXRvZ3JhZC5ncmFkKGYsIHgsIHRvcmNoLm9uZXNfbGlrZShmKSwgY3JlYXRlX2dyYXBoPVRydWUsIHJldGFpbl9ncmFwaD1UcnVlKVswXQ0KDQoNCmRlZiBjb2VmZnMoZXRhKToNCiAgICAiIiJwZXItcG9pbnQgZTMxLCBlMzM6IGJvdHRvbSAoZXRhPDAuNSkgK0UzMSwrRTMzIDsgdG9wIGZsaXBwZWQuIiIiDQogICAgc2cgPSB0b3JjaC53aGVyZShldGEgPCAwLjUsIDEuMCwgLTEuMCkNCiAgICByZXR1cm4gc2cgKiBFMzEsIHNnICogRTMzDQoNCg0KZGVmIG1ha2VfcG9vbHMoc2VlZCwgbl9jb2w9NDA5Niwgbl9iZD0yNTYpOg0KICAgIHNlZWRzID0gW2ludCh4LmdlbmVyYXRlX3N0YXRlKDEpWzBdKSBmb3IgeCBpbiBucC5yYW5kb20uU2VlZFNlcXVlbmNlKHNlZWQpLnNwYXduKDYpXQ0KICAgIGNvbCA9IHFtYy5Tb2JvbCgyLCBzY3JhbWJsZT1UcnVlLCBzZWVkPXNlZWRzWzBdKS5yYW5kb21fYmFzZTIoaW50KG1hdGgubG9nMihuX2NvbCkpKQ0KDQogICAgZGVmIHNpZGUoayk6DQogICAgICAgIHJldHVybiBxbWMuU29ib2woMSwgc2NyYW1ibGU9VHJ1ZSwgc2VlZD1zZWVkc1trXSkucmFuZG9tX2Jhc2UyKGludChtYXRoLmxvZzIobl9iZCkpKVs6LCAwXQ0KDQogICAgemVyb3MsIG9uZXMgPSBucC56ZXJvcyhuX2JkKSwgbnAub25lcyhuX2JkKQ0KICAgIHBvb2xzID0geyJjb2wiOiBjb2wsDQogICAgICAgICAgICAgImxlZnQiOiBucC5zdGFjayhbemVyb3MsIHNpZGUoMSldLCAxKSwgInJpZ2h0IjogbnAuc3RhY2soW29uZXMsIHNpZGUoMildLCAxKSwNCiAgICAgICAgICAgICAiYm90IjogbnAuc3RhY2soW3NpZGUoMyksIHplcm9zXSwgMSksICJ0b3AiOiBucC5zdGFjayhbc2lkZSg0KSwgb25lc10sIDEpfQ0KICAgIHJldHVybiB7azogdG9yY2gudGVuc29yKHYsIGR0eXBlPURULCBkZXZpY2U9REVWKSBmb3IgaywgdiBpbiBwb29scy5pdGVtcygpfQ0KDQoNCmRlZiByZXNpZHVhbF90ZXJtcyhtb2RlbCwgeGkpOg0KICAgIHhpID0geGkuY2xvbmUoKS5yZXF1aXJlc19ncmFkXyhUcnVlKQ0KICAgIG8gPSBtb2RlbCh4aSkNCiAgICB1aCwgdmgsIHBoLCBzeCwgc3osIHRhdSwgRHgsIER5ID0gW29bOiwgaTppICsgMV0gZm9yIGkgaW4gcmFuZ2UoOCldDQogICAgZ3UsIGd2LCBncCA9IGQodWgsIHhpKSwgZCh2aCwgeGkpLCBkKHBoLCB4aSkNCiAgICBleHggPSBTQ19VIC8gTCAqIGd1WzosIDA6MV0NCiAgICBleXkgPSBTQ19WIC8gSCAqIGd2WzosIDE6Ml0NCiAgICBleHkgPSAwLjUgKiAoU0NfVSAvIEggKiBndVs6LCAxOjJdICsgU0NfViAvIEwgKiBndls6LCAwOjFdKQ0KICAgIEV4ID0gLVYgLyBMICogZ3BbOiwgMDoxXQ0KICAgIEV5ID0gLVYgLyBIICogZ3BbOiwgMToyXQ0KICAgIGUzMSwgZTMzID0gY29lZmZzKHhpWzosIDE6Ml0pDQogICAgciA9IHsic3giOiAoU0NfUyAqIHN4IC0gKEMxMSAqIGV4eCArIEMxMiAqIGV5eSAtIGUzMSAqIEV5KSkgLyBTQ19TLA0KICAgICAgICAgInN6IjogKFNDX1MgKiBzeiAtIChDMTIgKiBleHggKyBDMjIgKiBleXkgLSBlMzMgKiBFeSkpIC8gU0NfUywNCiAgICAgICAgICJ0YXUiOiAoU0NfUyAqIHRhdSAtIEcgKiBleHkpIC8gU0NfUywNCiAgICAgICAgICJEeCI6IChTQ19EICogRHggLSBFUFMgKiBFeCkgLyBTQ19ELA0KICAgICAgICAgIkR5IjogKFNDX0QgKiBEeSAtIChlMzEgKiBleHggKyBlMzMgKiBleXkgKyBFUFMgKiBFeSkpIC8gU0NfRH0NCiAgICBnc3gsIGdzeiwgZ3QsIGdEeCwgZ0R5ID0gZChzeCwgeGkpLCBkKHN6LCB4aSksIGQodGF1LCB4aSksIGQoRHgsIHhpKSwgZChEeSwgeGkpDQogICAgclsiZGl2MSJdID0gKGdzeFs6LCAwOjFdIC8gTCArIGd0WzosIDE6Ml0gLyBIKSAqIEwgICMgZXF1aWxpYnJpdW0sIHNjYWxlZCBieSBTL0wNCiAgICByWyJkaXYyIl0gPSAoZ3RbOiwgMDoxXSAvIEwgKyBnc3pbOiwgMToyXSAvIEgpICogTA0KICAgIHJbImRpdkQiXSA9IChnRHhbOiwgMDoxXSAvIEwgKyBnRHlbOiwgMToyXSAvIEgpICogTA0KICAgIHJldHVybiByDQoNCg0KZGVmIGxvc3NfZm4obW9kZWwsIFApOg0KICAgIHIgPSByZXNpZHVhbF90ZXJtcyhtb2RlbCwgUFsiY29sIl0pDQogICAgcGRlID0gc3VtKHYuc3F1YXJlKCkubWVhbigpIGZvciB2IGluIHIudmFsdWVzKCkpDQogICAgbyA9IG1vZGVsKFBbInJpZ2h0Il0pDQogICAgYmMgPSBvWzosIDNdLnNxdWFyZSgpLm1lYW4oKSArIG9bOiwgNV0uc3F1YXJlKCkubWVhbigpICsgb1s6LCA2XS5zcXVhcmUoKS5tZWFuKCkgICMgZnJlZSBlbmQ6IHN4PXRhdT0wLCBEeD0wDQogICAgbyA9IG1vZGVsKFBbImxlZnQiXSkNCiAgICBiYyA9IGJjICsgb1s6LCA2XS5zcXVhcmUoKS5tZWFuKCkgICMgRHg9MCBhdCBjbGFtcGVkIGVuZA0KICAgIGZvciBrIGluICgiYm90IiwgInRvcCIpOg0KICAgICAgICBvID0gbW9kZWwoUFtrXSkNCiAgICAgICAgYmMgPSBiYyArIG9bOiwgNF0uc3F1YXJlKCkubWVhbigpICsgb1s6LCA1XS5zcXVhcmUoKS5tZWFuKCkgICMgZnJlZSBzdXJmYWNlczogc3o9dGF1PTANCiAgICByZXR1cm4gcGRlICsgYmMsIHsicGRlIjogZmxvYXQocGRlLmRldGFjaCgpKSwgImJjIjogZmxvYXQoYmMuZGV0YWNoKCkpfQ0KDQoNCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gc2NvcmluZw0KZGVmIHNjb3JlKG1vZGVsLCBmZW0pOg0KICAgIHhpID0gdG9yY2gudGVuc29yKG5wLnN0YWNrKFtmZW1bInh5Il1bOiwgMF0gLyBMLCBmZW1bInh5Il1bOiwgMV0gLyBIXSwgMSksIGR0eXBlPURULCBkZXZpY2U9REVWKQ0KICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOg0KICAgICAgICBvID0gbW9kZWwoeGkpLmRvdWJsZSgpLmNwdSgpLm51bXB5KCkNCiAgICAgICAgdGlwID0gZmxvYXQobW9kZWwodG9yY2gudGVuc29yKFtbMS4wLCAwLjBdXSwgZHR5cGU9RFQsIGRldmljZT1ERVYpKVswLCAxXSkgKiBTQ19WDQogICAgdSwgdiwgcCA9IG9bOiwgMF0gKiBTQ19VLCBvWzosIDFdICogU0NfViwgb1s6LCAyXSAqIFYNCg0KICAgIGRlZiByZWwoYSwgYik6DQogICAgICAgIHJldHVybiBmbG9hdChucC5saW5hbGcubm9ybShhIC0gYikgLyBucC5saW5hbGcubm9ybShiKSkNCg0KICAgIHJldHVybiB7InVfcmVsTDIiOiByZWwodSwgZmVtWyJ1Il0pLCAidl9yZWxMMiI6IHJlbCh2LCBmZW1bInYiXSksICJwaGlfcmVsTDIiOiByZWwocCwgZmVtWyJwaGkiXSksDQogICAgICAgICAgICAidGlwX3YiOiB0aXAsICJ0aXBfdl9mZW0iOiBmbG9hdChmZW1bInYiXVtucC5hcmdtaW4obnAuYWJzKGZlbVsieHkiXVs6LCAwXSAtIEwpICsgZmVtWyJ4eSJdWzosIDFdKV0pfQ0KDQoNCmRlZiBidWlsZChhcm0sIHdpZHRoLCBzZWVkKToNCiAgICB0b3JjaC5tYW51YWxfc2VlZChzZWVkKQ0KICAgIG5ldCA9IE1MUChbMiwgMTAwLCAyNTAsIDhdKSBpZiBhcm0gPT0gIm1scCIgZWxzZSBSQkZLQU4oMiwgd2lkdGgsIDgpDQogICAgcmV0dXJuIFBpZXpvKG5ldC50byhkZXZpY2U9REVWLCBkdHlwZT1EVCkpDQoNCg0KZGVmIHJ1bihhKToNCiAgICBvdXQgPSBQYXRoKGEub3V0KQ0KICAgIG91dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpDQogICAgeiA9IG5wLmxvYWQoYS5mZW0pDQogICAgZmVtID0ge2s6IHpba10gZm9yIGsgaW4gKCJ4eSIsICJ1IiwgInYiLCAicGhpIil9DQogICAgUCA9IG1ha2VfcG9vbHMoYS5zZWVkLCBhLm5fY29sLCBhLm5fYmQpDQogICAgbW9kZWwgPSBidWlsZChhLmFybSwgYS53aWR0aCwgYS5zZWVkKQ0KICAgIHBhcmFtcyA9IHN1bShwLm51bWVsKCkgZm9yIHAgaW4gbW9kZWwucGFyYW1ldGVycygpKQ0KICAgIG9wdCA9IHRvcmNoLm9wdGltLkFkYW0obW9kZWwucGFyYW1ldGVycygpLCBscj1hLmxyKQ0KICAgIGhpc3QgPSBbXQ0KICAgIHQwID0gdGltZS5wZXJmX2NvdW50ZXIoKQ0KICAgIGZvciBlcCBpbiByYW5nZSgxLCBhLmFkYW1fc3RlcHMgKyAxKToNCiAgICAgICAgb3B0Lnplcm9fZ3JhZChzZXRfdG9fbm9uZT1UcnVlKQ0KICAgICAgICBsLCBwYXJ0cyA9IGxvc3NfZm4obW9kZWwsIFApDQogICAgICAgIGwuYmFja3dhcmQoKQ0KICAgICAgICBvcHQuc3RlcCgpDQogICAgICAgIGlmIGVwICUgMTAwID09IDAgb3IgZXAgPT0gYS5hZGFtX3N0ZXBzOg0KICAgICAgICAgICAgaGlzdC5hcHBlbmQoeyJzdGFnZSI6ICJhZGFtIiwgInN0ZXAiOiBlcCwgImxvc3MiOiBmbG9hdChsLmRldGFjaCgpKSwgKipwYXJ0c30pDQogICAgcmVzID0geyJzdGF0dXMiOiBTVEFUVVMsICJhcm0iOiBhLmFybSwgInNlZWQiOiBhLnNlZWQsICJwYXJhbXMiOiBwYXJhbXMsICJhZGFtX3N0ZXBzIjogYS5hZGFtX3N0ZXBzLA0KICAgICAgICAgICAiYWRhbV9zZWNvbmRzIjogdGltZS5wZXJmX2NvdW50ZXIoKSAtIHQwLCAiZHR5cGUiOiBzdHIoRFQpLCAidnNwbGl0IjogVlNQTElULCAic2NvcmVfYWRhbV9maW5hbCI6IHNjb3JlKG1vZGVsLCBmZW0pLA0KICAgICAgICAgICAic2NpZW50aWZpY19hY2NlcHRlZCI6IEZhbHNlLCAicmVmZXJlbmNlX2FjY2VwdGVkIjogRmFsc2V9DQogICAgcHJpbnQoanNvbi5kdW1wcyh7ImFybSI6IGEuYXJtLCAic2VlZCI6IGEuc2VlZCwgInBhcmFtcyI6IHBhcmFtcywgImFkYW0iOiByZXNbInNjb3JlX2FkYW1fZmluYWwiXX0pLCBmbHVzaD1UcnVlKQ0KICAgIGxiID0gdG9yY2gub3B0aW0uTEJGR1MobW9kZWwucGFyYW1ldGVycygpLCBscj0xLjAsIG1heF9pdGVyPTI1LCBtYXhfZXZhbD0xMjUsIHRvbGVyYW5jZV9ncmFkPTFlLTEyLCB0b2xlcmFuY2VfY2hhbmdlPTAuMCwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgIGhpc3Rvcnlfc2l6ZT0xMDAsIGxpbmVfc2VhcmNoX2ZuPSJzdHJvbmdfd29sZmUiKQ0KDQogICAgZGVmIGNsb3N1cmUoKToNCiAgICAgICAgbGIuemVyb19ncmFkKHNldF90b19ub25lPVRydWUpDQogICAgICAgIGwsIF8gPSBsb3NzX2ZuKG1vZGVsLCBQKQ0KICAgICAgICBsLmJhY2t3YXJkKCkNCiAgICAgICAgcmV0dXJuIGwNCg0KICAgIGxhc3QgPSBmbG9hdChsb3NzX2ZuKG1vZGVsLCBQKVswXS5kZXRhY2goKSkNCiAgICBhY2MgPSB7azogdi5kZXRhY2goKS5jbG9uZSgpIGZvciBrLCB2IGluIG1vZGVsLnN0YXRlX2RpY3QoKS5pdGVtcygpfQ0KICAgIHQxID0gdGltZS5wZXJmX2NvdW50ZXIoKQ0KICAgIGl0LCBwbGF0ZWF1LCBzdGF0dXMgPSAwLCAwLCAibGJmZ3NfdGltZV9jYXAiDQogICAgd2hpbGUgdGltZS5wZXJmX2NvdW50ZXIoKSAtIHQxIDwgYS5sYmZnc19zZWNvbmRzIGFuZCBpdCA8IGEubGJmZ3NfbWF4X2l0ZXI6DQogICAgICAgIGIgPSBsYi5zdGF0ZVtsYi5fcGFyYW1zWzBdXS5nZXQoIm5faXRlciIsIDApDQogICAgICAgIHRyeToNCiAgICAgICAgICAgIGxiLnN0ZXAoY2xvc3VyZSkNCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOiAgIyBub3FhOiBCTEUwMDENCiAgICAgICAgICAgIHN0YXR1cyA9IGYibGJmZ3NfZXhjZXB0aW9uOnt0eXBlKGUpLl9fbmFtZV9ffSINCiAgICAgICAgICAgIGJyZWFrDQogICAgICAgIGN1ciA9IGZsb2F0KGxvc3NfZm4obW9kZWwsIFApWzBdLmRldGFjaCgpKQ0KICAgICAgICBpZiBub3QgbWF0aC5pc2Zpbml0ZShjdXIpIG9yIGN1ciA+IGxhc3QgKiAxLjU6DQogICAgICAgICAgICBzdGF0dXMgPSAibGJmZ3NfcmVqZWN0ZWQiDQogICAgICAgICAgICBicmVhaw0KICAgICAgICBkbiA9IGxiLnN0YXRlW2xiLl9wYXJhbXNbMF1dLmdldCgibl9pdGVyIiwgMCkgLSBiDQogICAgICAgIGltcCA9IChsYXN0IC0gY3VyKSAvIG1heChhYnMobGFzdCksIDFlLTMwKQ0KICAgICAgICBhY2MgPSB7azogdi5kZXRhY2goKS5jbG9uZSgpIGZvciBrLCB2IGluIG1vZGVsLnN0YXRlX2RpY3QoKS5pdGVtcygpfQ0KICAgICAgICBsYXN0ID0gY3VyDQogICAgICAgIGl0ICs9IGRuDQogICAgICAgIGhpc3QuYXBwZW5kKHsic3RhZ2UiOiAibGJmZ3MiLCAic3RlcCI6IGl0LCAibG9zcyI6IGN1cn0pDQogICAgICAgIGlmIGRuID09IDA6DQogICAgICAgICAgICBzdGF0dXMgPSAibGJmZ3NfemVyb19wcm9ncmVzcyINCiAgICAgICAgICAgIGJyZWFrDQogICAgICAgIHBsYXRlYXUgPSBwbGF0ZWF1ICsgMSBpZiBpbXAgPCAxZS0xMCBlbHNlIDANCiAgICAgICAgaWYgcGxhdGVhdSA+PSA1Og0KICAgICAgICAgICAgc3RhdHVzID0gImxiZmdzX3BsYXRlYXUiDQogICAgICAgICAgICBicmVhaw0KICAgIGVsc2U6DQogICAgICAgIHN0YXR1cyA9ICJsYmZnc19pdGVyX2NhcCIgaWYgaXQgPj0gYS5sYmZnc19tYXhfaXRlciBlbHNlICJsYmZnc190aW1lX2NhcCINCiAgICBtb2RlbC5sb2FkX3N0YXRlX2RpY3QoYWNjKQ0KICAgIHJlcy51cGRhdGUoeyJsYmZnc19zdGF0dXMiOiBzdGF0dXMsICJsYmZnc19pdGVyYXRpb25zIjogaXQsICJsYmZnc19zZWNvbmRzIjogdGltZS5wZXJmX2NvdW50ZXIoKSAtIHQxLA0KICAgICAgICAgICAgICAgICJzY29yZV9sYmZnc19maW5hbCI6IHNjb3JlKG1vZGVsLCBmZW0pfSkNCiAgICB0b3JjaC5zYXZlKHsic3RhdGVfZGljdCI6IG1vZGVsLnN0YXRlX2RpY3QoKX0sIG91dCAvICJjaGVja3BvaW50LWxiZmdzLWZpbmFsLnB0IikNCiAgICAob3V0IC8gImhpc3RvcnkuanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyhoaXN0KSkNCiAgICAob3V0IC8gInJ1bi5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlcywgaW5kZW50PTIpKQ0KICAgIHByaW50KGpzb24uZHVtcHMoeyJhcm0iOiBhLmFybSwgInNlZWQiOiBhLnNlZWQsICJsYmZncyI6IHJlc1sic2NvcmVfbGJmZ3NfZmluYWwiXSwgInN0YXR1cyI6IHN0YXR1cywgIml0ZXJzIjogaXR9KSwgZmx1c2g9VHJ1ZSkNCg0KDQpkZWYgc21va2UoYSk6DQogICAgIiIiVGllciAwOiBwb29scywgaGFyZCBCQ3MsIGZpbml0ZSBsb3NzL2dyYWRpZW50cywgc2hvcnQgZGVzY2VudC4iIiINCiAgICBQID0gbWFrZV9wb29scygxLCAyNTYsIDMyKQ0KICAgIGZvciBhcm0gaW4gKCJtbHAiLCAicmJmIik6DQogICAgICAgIG0gPSBidWlsZChhcm0sIGEud2lkdGgsIDEpDQogICAgICAgIHhpID0gdG9yY2gudGVuc29yKFtbMC4wLCAwLjNdLCBbMC41LCAwLjBdLCBbMC41LCAxLjBdXSwgZHR5cGU9RFQsIGRldmljZT1ERVYpDQogICAgICAgIG8gPSBtKHhpKQ0KICAgICAgICBhc3NlcnQgdG9yY2guYWxsY2xvc2Uob1swLCA6Ml0sIHRvcmNoLnplcm9zKDIsIGR0eXBlPURULCBkZXZpY2U9REVWKSksICJ1LHYgbXVzdCB2YW5pc2ggYXQgeD0wIg0KICAgICAgICBhc3NlcnQgYWJzKGZsb2F0KG9bMSwgMl0pKSA8IDFlLTEyIGFuZCBhYnMoZmxvYXQob1syLCAyXSkgLSAxLjApIDwgMWUtMTIsICJwaGkgbXVzdCBiZSAwIGF0IGJvdHRvbSBhbmQgViBhdCB0b3AiDQogICAgICAgIGwwLCBfID0gbG9zc19mbihtLCBQKQ0KICAgICAgICBsMC5iYWNrd2FyZCgpDQogICAgICAgIGcgPSB0b3JjaC5jYXQoW3AuZ3JhZC5mbGF0dGVuKCkgZm9yIHAgaW4gbS5wYXJhbWV0ZXJzKCldKQ0KICAgICAgICBhc3NlcnQgdG9yY2guaXNmaW5pdGUobDApIGFuZCB0b3JjaC5pc2Zpbml0ZShnKS5hbGwoKQ0KICAgICAgICBvcHQgPSB0b3JjaC5vcHRpbS5BZGFtKG0ucGFyYW1ldGVycygpLCBscj0xZS0zKQ0KICAgICAgICBmb3IgXyBpbiByYW5nZSg1MCk6DQogICAgICAgICAgICBvcHQuemVyb19ncmFkKCkNCiAgICAgICAgICAgIGwsIF8gPSBsb3NzX2ZuKG0sIFApDQogICAgICAgICAgICBsLmJhY2t3YXJkKCkNCiAgICAgICAgICAgIG9wdC5zdGVwKCkNCiAgICAgICAgYXNzZXJ0IGZsb2F0KGwpIDwgZmxvYXQobDApLCAibG9zcyBtdXN0IGRlY3JlYXNlIg0KICAgICAgICBwcmludChhcm0sICJPSyIsIGZsb2F0KGwwKSwgIi0+IiwgZmxvYXQobCksICJwYXJhbXMiLCBzdW0ocC5udW1lbCgpIGZvciBwIGluIG0ucGFyYW1ldGVycygpKSkNCg0KDQppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOg0KICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgibW9kZSIsIGNob2ljZXM9KCJzbW9rZSIsICJydW4iKSkNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tYXJtIiwgY2hvaWNlcz0oIm1scCIsICJyYmYiKSwgZGVmYXVsdD0icmJmIikNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0td2lkdGgiLCB0eXBlPWludCwgZGVmYXVsdD0xNikNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZHR5cGUiLCBjaG9pY2VzPSgiZmxvYXQzMiIsICJmbG9hdDY0IiksIGRlZmF1bHQ9ImZsb2F0NjQiKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS1kZXZpY2UiLCBkZWZhdWx0PSJjcHUiKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS12c3BsaXQiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAuMCkNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tc2VlZCIsIHR5cGU9aW50LCBkZWZhdWx0PTEwMDEpDQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWZlbSIpDQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW91dCIpDQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW4tY29sIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NDA5NikNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbi1iZCIsIHR5cGU9aW50LCBkZWZhdWx0PTI1NikNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tYWRhbS1zdGVwcyIsIHR5cGU9aW50LCBkZWZhdWx0PTMwMDApDQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWxyIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0xZS0zKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS1sYmZncy1zZWNvbmRzIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0zMDAuMCkNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbGJmZ3MtbWF4LWl0ZXIiLCB0eXBlPWludCwgZGVmYXVsdD0xNTAwMCkNCiAgICBhcmdzID0gYXAucGFyc2VfYXJncygpDQogICAgRFQgPSB0b3JjaC5mbG9hdDMyIGlmIGFyZ3MuZHR5cGUgPT0gImZsb2F0MzIiIGVsc2UgdG9yY2guZmxvYXQ2NA0KICAgIERFViA9IHRvcmNoLmRldmljZShhcmdzLmRldmljZSkNCiAgICBWU1BMSVQgPSBhcmdzLnZzcGxpdA0KICAgIChzbW9rZSBpZiBhcmdzLm1vZGUgPT0gInNtb2tlIiBlbHNlIHJ1bikoYXJncykNCg==', 'fem.npz': '...'}
for k, v in BLOB.items(): (p / k).write_bytes(base64.b64decode(v))
names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print(json.dumps({"gpus": names, "torch": torch.__version__, "hashes": {'piezo_pinn.py': '137faf7717d61d002d1e9a2c4d607cf608185741cbbec8cbc2f06f19cf3101cb', 'fem.npz': '39bc04a5ec51f71d229c96ffb011c5d32d96e1411a9733bbbc99178bda1e36e6'}}), flush=True)
assert names and all("T4" in x for x in names), names
JOBS = [{'arm': 'mlp', 'dtype': 'float64', 'seed': 1001, 'adam': 10000, 'lbfgs_s': 300, 'vsplit': 0.0001}, {'arm': 'rbf', 'dtype': 'float64', 'seed': 1001, 'adam': 10000, 'lbfgs_s': 300, 'vsplit': 0.0001}, {'arm': 'mlp', 'dtype': 'float64', 'seed': 1001, 'adam': 10000, 'lbfgs_s': 300, 'vsplit': 0.001}, {'arm': 'rbf', 'dtype': 'float64', 'seed': 1001, 'adam': 10000, 'lbfgs_s': 300, 'vsplit': 0.001}]
running, ledger, queue = {}, [], list(JOBS)
while queue or running:
    for g in list(running):
        pr, tag, st = running[g]
        if pr.poll() is not None:
            ledger.append({"job": tag, "rc": pr.returncode, "wall": time.monotonic() - st}); del running[g]
    if time.monotonic() - T0 > CUT: queue = []
    if time.monotonic() - T0 > CAP:
        for g, (pr, tag, st) in running.items(): pr.kill()
        ledger.append({"note": "cap reached, killed", "running": [t for _, t, _ in running.values()]}); break
    for g in range(len(names)):
        if g not in running and queue:
            j = queue.pop(0); tag = f"{j['arm']}-{j['dtype']}-vs{j.get('vsplit', 0.0)}-seed-{j['seed']}"
            cmd = [sys.executable, str(p / "piezo_pinn.py"), "run", "--arm", j["arm"], "--dtype", j["dtype"], "--seed", str(j["seed"]),
                   "--width", str(j.get("width", 16)), "--device", f"cuda:{g}", "--fem", str(p / "fem.npz"), "--out", str(w / "results" / tag),
                   "--vsplit", str(j.get("vsplit", 0.0)), "--adam-steps", str(j["adam"]), "--lbfgs-seconds", str(j["lbfgs_s"])]
            running[g] = (subprocess.Popen(cmd, stdout=open(w / f"{tag}.log", "w"), stderr=subprocess.STDOUT), tag, time.monotonic())
    time.sleep(2)
summ = {"wall": time.monotonic() - T0, "cap": CAP, "ledger": ledger, "gpus": names, "scientific_accepted": False, "reference_accepted": False}
(w / "session_summary.json").write_text(json.dumps(summ, indent=1)); print(json.dumps(summ))
